# Gemini Annotation Pipeline Evaluation: 5-Class Merged Prompting

This notebook evaluates the **Gemini Annotation Pipeline** (Few-Shot Strategy) on `data/human_dataset.csv` using a **pure 5-class prompt**.

### Fair Evaluation Setup:
To ensure a fair evaluation against baseline classifiers that operate on fewer target classes, the prompt provided to Gemini **completely excludes `Motivators`, `Interests`, and `Learnings`**. Gemini only sees and outputs the exact same 5 target categories as the comparison models:
1. **`Background`**
2. **`Achievements`**
3. **`Education`**
4. **`Work Experience`**
5. **`Others`**

Ground-truth human labels in `data/human_dataset.csv` are also mapped to these 5 categories (`Motivators`, `Interests`, `Learnings` $
ightarrow$ `Others`).


In [3]:
import os
import ast
import json
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import IPython.display as display

# Import workspace pipeline modules & 5-class taxonomy configs
import src.config as config
from src.prompt_builder import build_batch_classification_prompt
from src.gemini_client import get_batch_classification
from src.config import (
    CATEGORIES,
    CATEGORIES_5CLASS, 
    DEFINITIONS_5CLASS, 
    FEW_SHOT_EXAMPLES_5CLASS, 
    BATCH_SIZE
)

MERGE_MAP = {
    'background': 'Background',
    'achievements': 'Achievements',
    'education': 'Education',
    'work experience': 'Work Experience',
    'interests': 'Others',
    'motivators': 'Others',
    'learnings': 'Others',
    'others': 'Others'
}

def merge_ground_truth(lbl_list):
    """
    Maps ground truth human labels (Motivators, Interests, Learnings -> Others).
    """
    if isinstance(lbl_list, str):
        try:
            lbl_list = ast.literal_eval(lbl_list)
        except:
            lbl_list = [lbl_list]
            
    mapped = [MERGE_MAP.get(str(x).strip().lower(), 'Others') for x in lbl_list]
    return list(dict.fromkeys(mapped))

print('Setup complete.')
print(f'Active 5-Class Target Categories: {CATEGORIES_5CLASS}')


Setup complete.
Active 5-Class Target Categories: ['Background', 'Achievements', 'Education', 'Work Experience', 'Others']


In [ ]:
# 1. Load Human Ground-Truth Dataset
HUMAN_DATASET_PATH = 'data/human_dataset.csv'
df_human = pd.read_csv(HUMAN_DATASET_PATH)

df_human['raw_true_labels'] = df_human['labels'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_human['true_labels_5class'] = df_human['raw_true_labels'].apply(merge_ground_truth)

print(f'Loaded {len(df_human)} human-annotated sentences from {HUMAN_DATASET_PATH}.')
df_human[['doc_id', 'sent_id', 'text', 'raw_true_labels', 'true_labels_5class']].head(10)


In [ ]:
# 2. Pipeline Execution Engine (5-Class Prompting Mode)
def run_5class_few_shot_pipeline(df_input, batch_size=30, json_output_file='data/human_fewshot_5class_predictions.json', csv_output_file='data/human_fewshot_5class_predictions.csv', force_rerun=False):
    """
    Executes Gemini classification on human_dataset.csv prompting ONLY on the 5 target categories.
    """
    if not force_rerun and json_output_file and os.path.exists(json_output_file):
        print(f'Loading cached 5-class predictions from {json_output_file}...')
        with open(json_output_file, 'r', encoding='utf-8') as f:
            return json.load(f)
            
    original_strategy = config.PROMPTING_STRATEGY
    config.PROMPTING_STRATEGY = 'few_shot'
    
    items = []
    for idx, row in df_input.iterrows():
        items.append({
            'data': {
                'sent_id': int(row['sent_id']),
                'doc_id': int(row['doc_id']),
                'text': str(row['text']),
                'true_labels': row['true_labels_5class'],
                'title': str(row['title'])
            },
            'temp_id': idx
        })
        
    def chunk_list(l, n):
        for i in range(0, len(l), n):
            yield l[i:i + n]
            
    chunks = list(chunk_list(items, batch_size))
    results_list = []
    previous_sentence = None
    
    print(f'Running 5-Class Gemini Few-Shot Pipeline on {len(items)} sentences ({len(chunks)} batches)...')
    
    for batch_idx, batch in tqdm(enumerate(chunks), total=len(chunks), desc='5-Class Few-Shot Pipeline'):
        try:
            if batch[0]['data']['sent_id'] == 0:
                previous_sentence = None
                
            # Prompt builder with 5-class definitions, categories, and exemplars
            prompt = build_batch_classification_prompt(
                batch, 
                previous_sentence=previous_sentence,
                categories=CATEGORIES_5CLASS,
                definitions=DEFINITIONS_5CLASS,
                few_shot_examples=FEW_SHOT_EXAMPLES_5CLASS
            )
            
            # Gemini API call enforcing 5-class JSON response schema
            batch_results = get_batch_classification(batch, prompt, categories=CATEGORIES_5CLASS)
            
            res_map = {str(r.get('id')): r.get('c', ['Others']) for r in batch_results if r.get('id') is not None}
            
            for item in batch:
                s_id = str(item['data']['sent_id'])
                pred_c = res_map.get(s_id, ['Others'])
                if isinstance(pred_c, str):
                    pred_c = [pred_c]
                item_res = dict(item)
                item_res['data']['pred_labels'] = pred_c
                results_list.append(item_res)
                
            previous_sentence = batch[-1]['data']['text']
            time.sleep(0.5)
            
        except Exception as e:
            print(f'Batch {batch_idx} error: {e}')
            for item in batch:
                item_res = dict(item)
                item_res['data']['pred_labels'] = ['Others']
                results_list.append(item_res)
                
    config.PROMPTING_STRATEGY = original_strategy
    
    # Save JSON
    if json_output_file:
        with open(json_output_file, 'w', encoding='utf-8') as f:
            json.dump(results_list, f, indent=2, ensure_ascii=False)
        print(f'[✓] Saved 5-class predictions JSON to {json_output_file}')
        
    # Save CSV
    if csv_output_file:
        export_rows = []
        for res in results_list:
            d = res['data']
            t_lbls = d.get('true_labels', [])
            p_lbls = d.get('pred_labels', [])
            export_rows.append({
                'doc_id': d.get('doc_id'),
                'sent_id': d.get('sent_id'),
                'title': d.get('title'),
                'text': d.get('text'),
                'true_labels_5class': json.dumps(t_lbls),
                'pred_labels_5class': json.dumps(p_lbls),
                'exact_match': set(t_lbls) == set(p_lbls)
            })
        pd.DataFrame(export_rows).to_csv(csv_output_file, index=False)
        print(f'[✓] Exported 5-class predictions CSV to {csv_output_file}')
        
    return results_list


In [ ]:
# 3. Execute 5-Class Pipeline
# Set RUN_FULL = True to process all 3,661 sentences of human_dataset.csv
# Set FORCE_RERUN = True to ignore cache and run fresh API calls
RUN_FULL = False
FORCE_RERUN = True

eval_df = df_human.head(300) if not RUN_FULL else df_human.copy()
print(f'Running 5-class Gemini Few-Shot pipeline on {len(eval_df)} sentences...')

predictions = run_5class_few_shot_pipeline(
    eval_df, 
    batch_size=30, 
    json_output_file='data/human_fewshot_5class_predictions.json',
    csv_output_file='data/human_fewshot_5class_predictions.csv',
    force_rerun=FORCE_RERUN
)

y_true = [item['data']['true_labels'] for item in predictions]
y_pred = [item['data']['pred_labels'] for item in predictions]
print(f'Loaded {len(predictions)} prediction records under pure 5-class schema.')


In [ ]:
# 4. Calculate Merged Multi-Label Metrics (5 Classes)
mlb = MultiLabelBinarizer(classes=CATEGORIES_5CLASS)
y_true_bin = mlb.fit_transform(y_true)
y_pred_bin = mlb.transform(y_pred)

# Subset Accuracy (Exact Match Ratio)
subset_accuracy = accuracy_score(y_true_bin, y_pred_bin)

metrics_dict = {
    'Subset Accuracy (Exact Match)': subset_accuracy
}

for avg in ['micro', 'macro', 'weighted', 'samples']:
    metrics_dict[f'Precision ({avg.capitalize()})'] = precision_score(y_true_bin, y_pred_bin, average=avg, zero_division=0)
    metrics_dict[f'Recall ({avg.capitalize()})'] = recall_score(y_true_bin, y_pred_bin, average=avg, zero_division=0)
    metrics_dict[f'F1-Score ({avg.capitalize()})'] = f1_score(y_true_bin, y_pred_bin, average=avg, zero_division=0)

# Summary DataFrame
summary_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric', 'Few-Shot Score (Pure 5-Class)'])
summary_df['Percentage'] = summary_df['Few-Shot Score (Pure 5-Class)'].apply(lambda x: f'{x * 100:.2f}%')

print('=' * 70)
print('      GEMINI FEW-SHOT EVALUATION (PURE 5-CLASS PROMPT & SCHEMA)')
print('=' * 70)
try:
    print(summary_df.to_markdown(index=False))
except Exception:
    print(summary_df.to_string(index=False))
print('=' * 70)

# Display styled HTML table in Jupyter Notebook
display.display(summary_df.style.format({'Few-Shot Score (Pure 5-Class)': '{:.4f}'})
                .set_caption('Pure 5-Class Few-Shot Performance Metrics Summary'))


In [ ]:
# 5. Per-Category Detailed Performance Breakdown (5 Target Classes)
prec_per_cat = precision_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
rec_per_cat = recall_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
f1_per_cat = f1_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
support_per_cat = y_true_bin.sum(axis=0)

cat_breakdown_df = pd.DataFrame({
    'Category': CATEGORIES_5CLASS,
    'Ground Truth Count': support_per_cat,
    'Precision': [f'{p * 100:.2f}%' for p in prec_per_cat],
    'Recall': [f'{r * 100:.2f}%' for r in rec_per_cat],
    'F1-Score': [f'{f * 100:.2f}%' for f in f1_per_cat]
}).set_index('Category')

print('\n' + '=' * 70)
print('           PER-CATEGORY BREAKDOWN (PURE 5-CLASS SCHEMA)')
print('=' * 70)
try:
    print(cat_breakdown_df.to_markdown())
except Exception:
    print(cat_breakdown_df.to_string())
print('=' * 70)


--- 
## 📊 Standalone CSV Evaluation Cell (Offline Evaluation without API Calls)
Run the cell below to evaluate predictions directly from a `.csv` file. It automatically detects whether the file contains **5-class merged predictions** or **8-class unmerged predictions** and computes the appropriate non-merged/merged metrics.

In [5]:
# STANDALONE CSV EVALUATION CELL (Auto-detects 5-Class vs 8-Class Unmerged)
CSV_FILE_TO_EVALUATE = 'data/human_fewshot_predictions_unmerged.csv'  # Path to CSV predictions file

if not os.path.exists(CSV_FILE_TO_EVALUATE):
    if os.path.exists('data/human_fewshot_5class_predictions.csv'):
        CSV_FILE_TO_EVALUATE = 'data/human_fewshot_5class_predictions.csv'

if not os.path.exists(CSV_FILE_TO_EVALUATE):
    print(f'Error: File {CSV_FILE_TO_EVALUATE} not found. Check path or run the pipeline first.')
else:
    print(f'Evaluating directly from CSV file: {CSV_FILE_TO_EVALUATE}')
    df_eval = pd.read_csv(CSV_FILE_TO_EVALUATE)
    
    # Auto-detect if this is a 5-Class prediction or 8-Class unmerged prediction
    is_5class = ('5class' in CSV_FILE_TO_EVALUATE.lower()) or ('true_labels_5class' in df_eval.columns) or ('pred_labels_5class' in df_eval.columns)
    
    def parse_raw_list(val):
        if isinstance(val, str):
            try:
                return json.loads(val)
            except:
                try:
                    return ast.literal_eval(val)
                except:
                    return [val]
        return val if isinstance(val, list) else [val]
    
    if is_5class:
        print('-> Mode: 5-Class Merged Taxonomy Evaluation')
        target_cats = CATEGORIES_5CLASS
        true_col = 'true_labels_5class' if 'true_labels_5class' in df_eval.columns else 'true_labels'
        pred_col = 'pred_labels_5class' if 'pred_labels_5class' in df_eval.columns else ('pred_labels_merged' if 'pred_labels_merged' in df_eval.columns else 'pred_labels')
        
        def parse_col_labels(val):
            return merge_ground_truth(parse_raw_list(val))
    else:
        print('-> Mode: 8-Class Unmerged Taxonomy Evaluation (Original Categories)')
        target_cats = CATEGORIES
        true_col = 'raw_true_labels' if 'raw_true_labels' in df_eval.columns else ('true_labels' if 'true_labels' in df_eval.columns else df_eval.columns[0])
        pred_col = 'pred_labels_raw' if 'pred_labels_raw' in df_eval.columns else ('pred_labels' if 'pred_labels' in df_eval.columns else df_eval.columns[1])
        
        cat_map_8 = {c.lower(): c for c in CATEGORIES}
        def parse_col_labels(val):
            return [cat_map_8.get(str(x).strip().lower(), str(x).strip()) for x in parse_raw_list(val)]
    
    y_true_csv = df_eval[true_col].apply(parse_col_labels).tolist()
    y_pred_csv = df_eval[pred_col].apply(parse_col_labels).tolist()
    
    mlb_csv = MultiLabelBinarizer(classes=target_cats)
    y_true_bin_csv = mlb_csv.fit_transform(y_true_csv)
    y_pred_bin_csv = mlb_csv.transform(y_pred_csv)
    
    subset_acc_csv = accuracy_score(y_true_bin_csv, y_pred_bin_csv)
    
    csv_metrics = {'Subset Accuracy (Exact Match)': subset_acc_csv}
    for avg in ['micro', 'macro', 'weighted', 'samples']:
        csv_metrics[f'Precision ({avg.capitalize()})'] = precision_score(y_true_bin_csv, y_pred_bin_csv, average=avg, zero_division=0)
        csv_metrics[f'Recall ({avg.capitalize()})'] = recall_score(y_true_bin_csv, y_pred_bin_csv, average=avg, zero_division=0)
        csv_metrics[f'F1-Score ({avg.capitalize()})'] = f1_score(y_true_bin_csv, y_pred_bin_csv, average=avg, zero_division=0)
        
    csv_summary_df = pd.DataFrame(list(csv_metrics.items()), columns=['Metric', 'Score (From CSV)'])
    csv_summary_df['Percentage'] = csv_summary_df['Score (From CSV)'].apply(lambda x: f'{x * 100:.2f}%')
    
    print('=' * 70)
    print(f'          CSV EVALUATION RESULTS ({len(df_eval)} Samples)')
    print('=' * 70)
    try:
        print(csv_summary_df.to_markdown(index=False))
    except Exception:
        print(csv_summary_df.to_string(index=False))
    print('=' * 70)
    
    prec_csv = precision_score(y_true_bin_csv, y_pred_bin_csv, average=None, zero_division=0)
    rec_csv = recall_score(y_true_bin_csv, y_pred_bin_csv, average=None, zero_division=0)
    f1_csv = f1_score(y_true_bin_csv, y_pred_bin_csv, average=None, zero_division=0)
    supp_csv = y_true_bin_csv.sum(axis=0)
    
    csv_cat_df = pd.DataFrame({
        'Category': target_cats,
        'Support': supp_csv,
        'Precision': [f'{p * 100:.2f}%' for p in prec_csv],
        'Recall': [f'{r * 100:.2f}%' for r in rec_csv],
        'F1-Score': [f'{f * 100:.2f}%' for f in f1_csv]
    }).set_index('Category')
    
    print('\n' + '=' * 70)
    print('          CSV PER-CATEGORY PERFORMANCE BREAKDOWN')
    print('=' * 70)
    try:
        print(csv_cat_df.to_markdown())
    except Exception:
        print(csv_cat_df.to_string())
    print('=' * 70)
    
    display.display(csv_summary_df.style.format({'Score (From CSV)': '{:.4f}'})
                    .set_caption('CSV Prediction Evaluation Metrics Summary'))


Evaluating directly from CSV file: data/human_fewshot_predictions_unmerged.csv
-> Mode: 8-Class Unmerged Taxonomy Evaluation (Original Categories)
          CSV EVALUATION RESULTS (3661 Samples)
                       Metric  Score (From CSV) Percentage
Subset Accuracy (Exact Match)          0.783119     78.31%
            Precision (Micro)          0.803897     80.39%
               Recall (Micro)          0.842188     84.22%
             F1-Score (Micro)          0.822597     82.26%
            Precision (Macro)          0.611024     61.10%
               Recall (Macro)          0.846418     84.64%
             F1-Score (Macro)          0.678310     67.83%
         Precision (Weighted)          0.842873     84.29%
            Recall (Weighted)          0.842188     84.22%
          F1-Score (Weighted)          0.832285     83.23%
          Precision (Samples)          0.823295     82.33%
             Recall (Samples)          0.843212     84.32%
           F1-Score (Samples)         